In [1]:
from collections import Counter, defaultdict
import json
import pandas as pd
from pathlib import Path

In [2]:
dataset_path = Path("/Users/fangzhouma/Desktop/3d_vision/3D-VLM-benchmark-OOS/outputs/jsonl_v2/multi_turn.jsonl")
# Change this path if your JSONL is elsewhere.

raw_docs = []
with dataset_path.open("r") as f:
    for line in f:
        raw_docs.append(json.loads(line))

len(raw_docs), raw_docs[0].keys()

(110,
 dict_keys(['trajectory_id', 'question_class', 'video_id', 'video_path', 'query_time_sec', 'query_time_in_clip_sec', 'clip_start_time_sec', 'clip_end_time_sec', 'clip_duration_sec', 'horizon_sec', 'object_a_assoc_id', 'object_a_name', 'num_incremental_steps', 'num_branch_steps', 'terminated_at_step', 'stop_reason', 'generation_info', 'doc_id', 'mode', 'steps', 'include_gold_history', 'gold_history_messages']))

In [3]:
rows = []

for raw in raw_docs:
    for step in raw.get("steps", []):
        if step.get("skipped"):
            continue

        rows.append({
            "trajectory_id": raw.get("trajectory_id"),
            "video_id": raw.get("video_id"),
            "doc_id": raw.get("doc_id"),
            "mode": raw.get("mode"),
            "query_time_sec": raw.get("query_time_sec"),
            "step": str(step.get("step")),
            "branch_group": step.get("branch_group"),
            "depends_on_steps": step.get("depends_on_steps"),
            "question_class": step.get("step_question_class"),
            "question": step.get("question"),
            "choices": step.get("choices") or [],
            "correct_idx": step.get("correct_idx"),
            "target_text": step.get("target_text"),
            "acceptable_idxs": step.get("acceptable_idxs"),
            "answer_metadata": step.get("answer_metadata"),
        })

df = pd.DataFrame(rows)
df.head()

,trajectory_id,video_id,doc_id,mode,query_time_sec,step,branch_group,depends_on_steps,question_class,question,choices,correct_idx,target_text,acceptable_idxs,answer_metadata
0,P01-20240202-110250__oos_staged_h5p0_0,P01-20240202-110250,P01-20240202-110250__oos_staged_h5p0_0,multi_turn,66.0,1,NaN,[],oos_step1_visibility,"At the current time <TIME 00:01:06.0 video 1>,...","[No, Yes]",0.0,No,None,"{'status': 'out_of_view', 'is_visible': False,..."
1,P01-20240202-110250__oos_staged_h5p0_0,P01-20240202-110250,P01-20240202-110250__oos_staged_h5p0_0,multi_turn,66.0,2,NaN,[1],oos_step2_last_visible,When was the previously moved third orange las...,[],NaN,"<TIME 00:01:00.0 video 1>; Point=(0.6107, 0.7685)",None,"{'sampled_last_visible_time_sec': 60.0, 'sampl..."
2,P01-20240202-110250__oos_staged_h5p0_0,P01-20240202-110250,P01-20240202-110250__oos_staged_h5p0_0,multi_turn,66.0,3,NaN,"[1, 2]",oos_step3_last_placement,At what time did the previously moved third or...,[],NaN,"<TIME 00:00:58.2 video 1>; Point=(0.5579, 0.7974)",None,"{'last_placement_time_sec': 58.2, 'last_placem..."
3,P01-20240202-110250__oos_staged_h5p0_0,P01-20240202-110250,P01-20240202-110250__oos_staged_h5p0_0,multi_turn,66.0,4,NaN,"[1, 2, 3]",oos_step4_fixture,"At the current time <TIME 00:01:06.0 video 1>,...","[drawer, counter, shelf, cupboard, bin]",1.0,counter,None,"{'reference_time_sec': 60.0, 'correct_fixture'..."
4,P01-20240202-110250__oos_staged_h5p0_0,P01-20240202-110250,P01-20240202-110250__oos_staged_h5p0_0,multi_turn,66.0,5a,post_step4,"[1, 2, 3, 4]",oos_branch_object_camera_relative_position,"At the current time <TIME 00:01:06.0 video 1>,...","[Back-right, Front-right, Back-left, Front-left]",1.0,Front-right,None,"{'reference_time_sec': 66.0, 'camera_coordinat..."


In [4]:
for qclass, g in df.groupby("question_class"):
    print("\n" + "=" * 100)
    print(qclass)
    print("N:", len(g))

    print("\nTarget text distribution:")
    print(g["target_text"].value_counts(dropna=False))

    print("\nCorrect idx distribution:")
    print(g["correct_idx"].value_counts(dropna=False))

    print("\nChoice order distribution:")
    print(g["choices"].apply(tuple).value_counts().head(10))


oos_branch_object_camera_relative_position
N: 110

Target text distribution:
target_text
Back-right     52
Back-left      24
Front-right    18
Front-left     16
Name: count, dtype: int64

Correct idx distribution:
correct_idx
1.0    31
2.0    27
3.0    26
0.0    26
Name: count, dtype: int64

Choice order distribution:
choices
(Back-right, Front-right, Front-left, Back-left)    8
(Front-right, Back-right, Front-left, Back-left)    8
(Back-right, Front-right, Back-left, Front-left)    7
(Front-left, Back-right, Back-left, Front-right)    7
(Back-left, Front-left, Back-right, Front-right)    7
(Front-left, Back-left, Front-right, Back-right)    7
(Front-right, Back-left, Back-right, Front-left)    6
(Back-right, Back-left, Front-right, Front-left)    6
(Back-right, Front-left, Back-left, Front-right)    6
(Front-right, Front-left, Back-left, Back-right)    5
Name: count, dtype: int64

oos_branch_object_object_distance
N: 110

Target text distribution:
target_text
medium        79
close  

In [ ]:
baseline_rows = []

for qclass, g in df.groupby("question_class"):
    n = len(g)

    semantic_counts = g["target_text"].value_counts(dropna=False)
    idx_counts = g["correct_idx"].value_counts(dropna=False)

    semantic_majority_acc = semantic_counts.iloc[0] / n
    idx_majority_acc = idx_counts.iloc[0] / n

    baseline_rows.append({
        "question_class": qclass,
        "n": n,
        "semantic_majority": semantic_counts.index[0],
        "semantic_majority_acc": semantic_majority_acc,
        "idx_majority": idx_counts.index[0],
        "idx_majority_acc": idx_majority_acc,
    })

baseline_df = pd.DataFrame(baseline_rows).sort_values("question_class")
baseline_df




,question_class,n,semantic_majority,semantic_majority_acc,idx_majority,idx_majority_acc
0,oos_branch_object_camera_relative_position,110,Back-right,0.472727,1.0,0.281818
1,oos_branch_object_object_distance,110,medium,0.718182,2.0,0.300000
2,oos_branch_object_object_relation,110,Back-right,0.600000,3.0,0.600000
3,oos_step1_visibility,110,No,1.000000,0.0,0.654545
4,oos_step2_last_visible,110,"<TIME 00:01:03.0 video 1>; Point=(0.1006, 0.7887)",0.018182,NaN,1.000000
5,oos_step3_last_placement,110,"<TIME 00:02:18.4 video 1>; Point=(0.6612, 0.7427)",0.072727,NaN,1.000000
6,oos_step4_fixture,110,counter,0.863636,1.0,0.245455


In [7]:
step1 = df[df["question_class"] == "oos_step1_visibility"].copy()

step1["choices_tuple"] = step1["choices"].apply(tuple)

step1[["choices_tuple", "correct_idx", "target_text"]].value_counts()

choices_tuple  correct_idx  target_text
(No, Yes)      0.0          No             72
(Yes, No)      1.0          No             38
Name: count, dtype: int64